In [1]:
import pandas as pd
from pathlib import Path

DF_PATH = Path("../data/processed/ames_housing_cleaned.parquet")
df = pd.read_parquet(DF_PATH)

In [2]:
# look at the cleaned dataframe
df.head()

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,None,IR1,Lvl,...,0,None,None,None,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,None,Reg,Lvl,...,0,None,MnPrv,None,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,None,IR1,Lvl,...,0,None,None,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,None,Reg,Lvl,...,0,None,None,None,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,None,IR1,Lvl,...,0,None,MnPrv,None,0,3,2010,WD,Normal,189900


In [3]:
# house age
df["House Age"] = df["Yr Sold"] - df["Year Built"]

In [4]:
# house remodelled
df["Was Remodelled"] = (df["Year Remod/Add"] > df["Year Built"]).astype(int)

In [5]:
# age since last remodel
df['Remod Age'] = df['Yr Sold'] - df['Year Remod/Add']

In [6]:
# total bathrooms
df['Total Bathrooms'] = df['Full Bath'] + 0.5 * df['Half Bath'] + df['Bsmt Full Bath'] + 0.5 * df['Bsmt Half Bath']

In [7]:
# capturing total outdoor living space
df['Total Porch SF'] = df['Open Porch SF'] + df['Enclosed Porch'] + df['3Ssn Porch'] + df['Screen Porch'] + df['Wood Deck SF']

In [8]:
# interaction features that can improve predictive power (useful for tree-based models or linear models)
df['Overall Qual Gr Liv'] = df['Overall Qual'] * df['Gr Liv Area']
df['Total Rooms Gr Liv'] = df['TotRms AbvGrd'] * df['Gr Liv Area']

In [9]:
# temporal features
df['Built Decade'] = (df['Year Built'] // 10) * 10

In [10]:
# has pool
df["Has Pool"] = (df["Pool Area"] > 0).astype(int)

In [11]:
# has fireplace
df["Has Fireplace"] = (df["Fireplaces"] > 0).astype(int)

In [12]:
# move target to end
col_to_move = df.pop("SalePrice")
df.insert(len(df.columns), "SalePrice", col_to_move)

In [13]:
# look at the final dataframe
df.head()

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Was Remodelled,Remod Age,Total Bathrooms,Total Porch SF,Overall Qual Gr Liv,Total Rooms Gr Liv,Built Decade,Has Pool,Has Fireplace,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,None,IR1,Lvl,...,0,50,2.0,272,9936,11592,1960,0,1,215000
1,2,526350040,20,RH,80.0,11622,Pave,None,Reg,Lvl,...,0,49,1.0,260,4480,4480,1960,0,0,105000
2,3,526351010,20,RL,81.0,14267,Pave,None,IR1,Lvl,...,0,52,1.5,429,7974,7974,1950,0,0,172000
3,4,526353030,20,RL,93.0,11160,Pave,None,Reg,Lvl,...,0,42,3.5,0,14770,16880,1960,0,1,244000
4,5,527105010,60,RL,74.0,13830,Pave,None,IR1,Lvl,...,1,12,2.5,246,8145,9774,1990,0,1,189900


In [14]:
# save final feature engineered dataset
df.to_parquet("../data/processed/ames_housing_featured_engineered.parquet", index=False)